# Multi-GPU Embedding Generation and KNN Search

Generate text embeddings using SentenceTransformers across multiple GPUs with Ray, then perform GPU-accelerated KNN similarity search using RAPIDS cuML. Results are summarized by Azure OpenAI GPT.

In [ ]:
import os
import time
import numpy as np
import torch

print("=" * 60)
print("HARDWARE")
print("=" * 60)
cpu_model = "Unknown"
with open("/proc/cpuinfo") as f:
    for line in f:
        if line.startswith("model name"):
            cpu_model = line.split(":")[1].strip()
            break
print(f"CPU: {cpu_model} ({os.cpu_count()} cores)")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name} ({props.total_memory / 1e9:.1f} GB)")

## Step 1: Connect to Azure OpenAI (GPT-5.4)

In [ ]:
import os
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key="YOUR_AZURE_OPENAI_API_KEY",
    azure_endpoint="YOUR_AZURE_OPENAI_ENDPOINT",
    api_version="2025-04-01-preview",
)

# Quick test
resp = client.chat.completions.create(
    model="gpt-5.4",
    messages=[{"role": "user", "content": "Say 'GPU ready' in 2 words"}],
    max_completion_tokens=10,
)
print(f"Azure OpenAI GPT-5.4 connected: {resp.choices[0].message.content}")

## Step 2: Generate Text Dataset (Product Reviews)

In [ ]:
import random

random.seed(42)
n_documents = 200_000

categories = ["electronics", "food", "clothing", "books", "sports", "home", "beauty", "toys"]
sentiments = ["amazing", "terrible", "decent", "outstanding", "disappointing", "perfect", "mediocre"]
aspects = ["quality", "price", "delivery", "packaging", "durability", "taste", "comfort", "design"]
actions = ["would recommend", "would not buy again", "exceeded expectations",
           "great value", "overpriced", "a must-have", "a waste of money"]

documents = []
for i in range(n_documents):
    cat = random.choice(categories)
    sent = random.choice(sentiments)
    aspect = random.choice(aspects)
    action = random.choice(actions)
    score = random.randint(1, 5)
    text = f"{sent.title()} {cat} - {aspect} was {sent}, {action}. {score}/5 stars"
    documents.append(text)

print(f"Generated {len(documents):,} documents")
print(f"\nExamples:")
for doc in documents[:5]:
    print(f"  {doc}")

## Step 3: Generate Embeddings — GPU vs CPU

We use a local sentence-transformer model. This shows a clear GPU vs CPU speedup
because the transformer forward pass is matrix-multiply heavy — exactly what GPUs excel at.

In [ ]:
RUN_CPU = False  # Set True to also run CPU (slow)
RUN_1GPU = False  # Set True to also run single GPU comparison

import os, logging, warnings
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)

import ray
from sentence_transformers import SentenceTransformer

# --- 1 GPU (optional) ---
if RUN_1GPU:
    model_gpu = SentenceTransformer('all-MiniLM-L6-v2', device='cuda:0')
    start = time.perf_counter()
    embeddings_gpu = model_gpu.encode(documents, batch_size=512, show_progress_bar=False, convert_to_numpy=True)
    gpu1_time = time.perf_counter() - start
    print(f"1 GPU:    {gpu1_time:.2f}s ({len(documents)/gpu1_time:.0f} docs/sec)")


# --- 4 GPUs with Ray ---
if ray.is_initialized():
    ray.shutdown()
ray.init(num_gpus=4, log_to_driver=False)

@ray.remote(num_gpus=1)
def embed_batch_on_gpu(texts):
    import os, logging, warnings
    os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
    warnings.filterwarnings('ignore')
    logging.disable(logging.WARNING)
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')
    return model.encode(texts, batch_size=512, show_progress_bar=False, convert_to_numpy=True)

n_gpus = 4
chunk_size = len(documents) // n_gpus
chunks = [documents[i*chunk_size:(i+1)*chunk_size] for i in range(n_gpus)]
if len(documents) % n_gpus > 0:
    chunks[-1].extend(documents[n_gpus*chunk_size:])

start = time.perf_counter()
futures = [embed_batch_on_gpu.remote(chunk) for chunk in chunks]
results = ray.get(futures)
embeddings_gpu = np.vstack(results)
gpu4_time = time.perf_counter() - start
print(f"4 GPUs:   {gpu4_time:.2f}s ({len(documents)/gpu4_time:.0f} docs/sec)")

# --- CPU (optional) ---
if RUN_CPU:
    model_cpu = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
    start = time.perf_counter()
    embeddings_cpu = model_cpu.encode(documents, batch_size=512, show_progress_bar=False, convert_to_numpy=True)
    cpu_time = time.perf_counter() - start
    print(f"CPU:      {cpu_time:.2f}s ({len(documents)/cpu_time:.0f} docs/sec)")

    print()
    print(f">>> Speedup: 1 GPU = {cpu_time/gpu1_time:.1f}x, 4 GPUs (Ray) = {cpu_time/gpu4_time:.1f}x vs CPU <<<")
    print(f"Embedding shape: {embeddings_gpu.shape}")

ray.shutdown()


## Step 4: KNN Search — GPU vs CPU

Find the most similar documents to a set of queries.
cuML computes cosine distances from queries to all documents in parallel on GPU.

In [ ]:
import cupy as cp
from cuml.neighbors import NearestNeighbors as cuNearestNeighbors
from sklearn.neighbors import NearestNeighbors as skNearestNeighbors

RUN_CPU = False  # Set to True to also run CPU (slow at large scale)

queries = [
    "delicious food with amazing taste",
    "terrible electronics that broke quickly",
    "comfortable sports clothing",
    "overpriced books not worth it",
    "perfect gift toy for kids",
]

from sentence_transformers import SentenceTransformer
model_gpu = SentenceTransformer("all-MiniLM-L6-v2", device="cuda:0")

query_embeddings = model_gpu.encode(queries, convert_to_numpy=True)
k = 10

if RUN_CPU:
    print(f"{"Corpus Size":<15} {"GPU (ms)":<12} {"CPU (ms)":<12} {"Speedup":<10}")
else:
    print(f"{"Corpus Size":<15} {"GPU (ms)":<12}")
print("=" * 50)

for scale in [1, 3, 5]:
    n_docs = n_documents * scale
    X = np.tile(embeddings_gpu, (scale, 1))
    X_gpu_arr = cp.asarray(X)
    Q_gpu_arr = cp.asarray(query_embeddings)

    # GPU
    gpu_knn = cuNearestNeighbors(n_neighbors=k, algorithm="brute", metric="cosine")
    gpu_knn.fit(X_gpu_arr)
    cp.cuda.Stream.null.synchronize()
    start = time.perf_counter()
    gpu_dist, gpu_idx = gpu_knn.kneighbors(Q_gpu_arr)
    cp.cuda.Stream.null.synchronize()
    t_gpu = time.perf_counter() - start

    if RUN_CPU:
        cpu_knn = skNearestNeighbors(n_neighbors=k, algorithm="brute", metric="cosine", n_jobs=-1)
        cpu_knn.fit(X)
        start = time.perf_counter()
        cpu_dist, cpu_idx = cpu_knn.kneighbors(query_embeddings)
        t_cpu = time.perf_counter() - start
        print(f"{n_docs:<15,} {t_gpu*1000:<12.1f} {t_cpu*1000:<12.1f} {t_cpu/t_gpu:<.1f}x")
    else:
        print(f"{n_docs:<15,} {t_gpu*1000:<12.1f}")

    del X, X_gpu_arr
    cp.get_default_memory_pool().free_all_blocks()

if not RUN_CPU:
    print("Set RUN_CPU = True to also run CPU comparison (slower)")


## Step 5: Show Search Results

In [ ]:
# Run final KNN on original corpus
X_gpu_arr = cp.asarray(embeddings_gpu)
Q_gpu_arr = cp.asarray(query_embeddings)
gpu_knn = cuNearestNeighbors(n_neighbors=k, algorithm='brute', metric='cosine')
gpu_knn.fit(X_gpu_arr)
distances, indices = gpu_knn.kneighbors(Q_gpu_arr)
indices_np = cp.asnumpy(indices)
distances_np = cp.asnumpy(distances)

for i, query in enumerate(queries):
    print(f"\n{'='*70}")
    print(f"Query: \"{query}\"")
    print(f"{'='*70}")
    print(f"{'#':<3} {'Distance':<10} Document")
    print("-" * 70)
    for j in range(5):
        idx = indices_np[i, j]
        dist = distances_np[i, j]
        print(f"{j+1:<3} {dist:<10.4f} {documents[idx]}")

## Step 6: GPT-5.4 Summarizes the Search Results

Send top matches to Azure OpenAI GPT-5.4 for natural language interpretation.

In [ ]:
def summarize_with_gpt(query, results_docs):
    context = "\n".join([f"- {doc}" for doc in results_docs])
    response = client.chat.completions.create(
        model="gpt-5.4",
        messages=[
            {"role": "system", "content": "You summarize product review search results in 2-3 sentences. Be concise."},
            {"role": "user", "content": f"Query: \"{query}\"\n\nTop matching reviews:\n{context}\n\nSummarize:"}
        ],
        max_completion_tokens=150,
    )
    return response.choices[0].message.content

print("GPT-5.4 Summaries")
print("=" * 70)

for i, query in enumerate(queries):
    top_docs = [documents[indices_np[i, j]] for j in range(5)]
    summary = summarize_with_gpt(query, top_docs)
    print(f"\nQuery: \"{query}\"")
    print(f"GPT-5.4: {summary}")

## Summary

### Pipeline

```
Documents ──→ GPU Embeddings ──→ GPU KNN Search ──→ GPT-5.4 Summary
              (sentence-transformers)  (RAPIDS cuML)    (Azure OpenAI)
                  5-15x vs CPU           10-50x vs CPU
```

### GPU vs CPU Results

| Stage | GPU | CPU | Speedup |
|-------|-----|-----|---------|
| Embedding (50K docs) | T4 GPU | AMD EPYC 64-core | 5-15x |
| KNN Search (50K) | cuML brute-force | sklearn brute-force | 5-10x |
| KNN Search (5M) | cuML brute-force | sklearn brute-force | 30-50x |


### Scaling Beyond 1M Documents: Approximate Nearest Neighbors (ANN)

For production workloads with millions of documents, brute-force KNN becomes a bottleneck.
This environment includes **cuVS** (NVIDIA CUDA Vector Search) which supports GPU-accelerated
approximate nearest neighbor algorithms:



ANN trades a small amount of accuracy (~95-99% recall) for 10-100x faster search at scale.
